# 文档加载
LangChain提供了丰富的文档加载器（Document Loaders），支持从各种来源加载文档，例如：
- Webpages: 将网页内容加载为Document，例如WebBaseLoader
- PDFs: 将PDF文件加载为Document，例如PyPDF
- CommonFiles: 各种常见文件类型加载为Document，例如TextLoader、CSVLoader
- Social platforms: 从社交媒体加载文档，例如Twitter、Reddit
- Messaging services: 从消息平台加载文档，例如Telegram、WhatsApp、Discord
- Productivity tools: 从常用的生产力工具中加载文档，例如Figma、Github、Slack


虽然加载器各不相同，但都实现了BaseLoader接口，因此都具有两个通用方法：
- load() : 一次性加载所有文档
- lazy_load() : 基于流式传输懒加载文档，适用于大数据集
所有加载器都将原始数据转换为统一的 Document 对象，包含：
- page_content: 文档内容
- metadata: 元数据（如来源、页码等）

## TextLoader
TextLoader是社区提供的加载器，作用是加载普通的txt文件，这也是最常见的一种文本文件类型，格式简单

In [1]:
from langchain_community.document_loaders import TextLoader

with open("sample.txt", "w", encoding="utf-8") as f:
    f.write("LangChain是用于构建LLM应用的框架。\n")
    f.write("LangGraph是LangChain的图结构编排库。\n")
    f.write("LangSmith是调试监控平台。\n")

In [ ]:
# 加载文本文件
loader = TextLoader("sample.txt", encoding="utf-8")
docs = loader.load()

print(f"加载了{len(docs)}个文档")
print(f"内容：{docs[0].page_content}")
print(f"元数据：{docs[0].metadata}")

加载了1
内容：LangChain是用于构建LLM应用的框架。
LangGraph是LangChain的图结构编排库。
LangSmith是调试监控平台。

元数据：{'source': 'sample.txt'}


## WebBaseLoader
WebBaseLoader同样是社区提供的加载器，只要给一个url地址，它就能自动读取网页内容，去掉无用的Html、CSS、JS元素，只保留普通文本数据。

In [4]:
from langchain_community.document_loaders import WebBaseLoader

# 加载网页内容
loader = WebBaseLoader(
    web_path=["https://docs.langchain.com/oss/python/langchain/rag"]
)

docs = loader.load()

print(f"加载了 {len(docs)} 个文档")
print(f"来源: {docs[0].metadata.get('source', 'unknown')}")
print(f"内容长度: {len(docs[0].page_content)} 字符")
print(f"内容预览: {docs[0].page_content[:200]}...")

加载了 1 个文档
来源: https://docs.langchain.com/oss/python/langchain/rag
内容长度: 82878 字符
内容预览: Retrieval Augmented Generation (RAG) with Deep Agents - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exp...


## CSVLoader
CSVLoader也是社区提供的加载器，它可以加载csv格式的文件

In [5]:
from langchain_community.document_loaders.csv_loader import CSVLoader

import csv
with open("sample.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["name", "description", "category"])
    writer.writerow(["LangChain", "LLM应用开发框架", "AI框架"])
    writer.writerow(["LangGraph", "图结构编排库", "AI框架"])
    writer.writerow(["LangSmith", "调试监控平台", "AI工具"])

In [6]:
# 加载CSV文件
loader = CSVLoader(
    file_path="sample.csv",
    source_column="name",
    encoding="utf-8"
)

docs = loader.load()

print(f"加载了{len(docs)}行数据")
for doc in docs:
    print(f"[{doc.metadata.get('source', '?')}] {doc.page_content[:100]}")

加载了3行数据
[LangChain] name: LangChain
description: LLM应用开发框架
category: AI框架
[LangGraph] name: LangGraph
description: 图结构编排库
category: AI框架
[LangSmith] name: LangSmith
description: 调试监控平台
category: AI工具


## PyPDFLoader
PyPDFLoader顾名思义，是加载PDF文件的加载器，依赖于pypdf

PyPDFLoader支持两种模式：
- single：整个文档作为一个Document，但是可以自定义文档页与页之间的分隔符
- page：每页作为一个Document

In [8]:
from langchain_community.document_loaders import PyPDFLoader

# 加载PDF文件
loader = PyPDFLoader(
    "sample.pdf",
    mode="single", # single \ page
    pages_delimiter="\n-------THIS IS A CUSTOM END OF PAGE-------\n" # 自定义页分隔符，可选
)

docs = loader.load()

# 清洗 PDF 中可能存在的非法 Unicode 代理字符
for doc in docs:
    doc.page_content = doc.page_content.encode("utf-8", errors="replace").decode("utf-8")

print(f"加载了 {len(docs)} 页")
print("-"*50)
print(f"第1页内容: {docs[0].page_content[:15800]}...")
print("-"*50)
print(f"元数据: {docs[0].metadata}")  # 包含 source, page 等信息

加载了 1 页
--------------------------------------------------
第1页内容: 基于历史数据预测未来股价收益
本次竞赛的目标是基于沪深 300 指数成分股的历史股价数据，通过建立机器学习模
型来预测未来一周收益最大的股票组合（不超过 5 只，累计占比不超过 1）。选手需通过
构建模型、训练和调优，预测并输出未来一周沪深 300 指数成分股收益最大的股票组合。
一、 比赛数据
1) 本次比赛不限制数据格式、内容、范围等，只要所有人都可以免费公开下载使
用即可；
2) 大赛基准代码（https://github.com/Sherlock1956/THU-BDC2026）给出的数据
为通过 baostock 平台下载的数据；位于 data 目录，相关说明详见：
get_stock_data_illustration.txt。
二、 输出结果
选手的任务是训练模型，基于已有数据，输出预测结果 result.csv（UTF-8 编码），
示例如下：
stock_id,weight
000408,0.2
000975,0.2
002028,0.2
600372,0.2
600036,0.2
1) 带有表头：stock_id,weight；
2) 给出不超过 5 个不同的股票代码及权重，每行一个代码及其权重，权重累加和
不超过 1，不到 1 的部分意味着持有现金；
3) 每行都用‘,’分割。
-------THIS IS A CUSTOM END OF PAGE-------
三、 评估标准
1. 计算收益率：
为了计算这个投资组合从 T+1 开盘买入到 T+5 开盘卖出的总收益率，我们需要先
计算出每只股票的单票收益率，然后将它们按权重进行加权求和。由于现金部分通常默
认不产生收益（收益率为0），剩余权重即为现金保留。
以下是详细的计算公式：
1) 变量定义
 n：投资组合中的股票总数（n ≤ 5）。
 wi：第 i 只股票的分配权重。
 Pi,T+1
open：第 i 只股票在 T+1 日的开盘价（买入价）。
 Pi,T+5
open：第 i 只股票在 T+5 日的开盘价（卖出价）。
 wcash：现金权重，计算方式为wcash = 1 − i=1
n wi  。
2) 单只股票的收益率公式
对于第 i 只股票，其

## 复杂文本加载工具
### PDF处理高级工具

#### 基于LangChain

In [10]:
from langchain_mineru import MinerULoader

# 初始化客户端
loader = MinerULoader(
    source="sample.pdf",
    mode="flash"
)

# 解析文档，返回值直接是LangChain的Document集合
docs = loader.load()

print(docs[0].metadata) # 元数据
print(docs[0].page_content) # 文档内容

{'source': 'sample.pdf', 'loader': 'mineru', 'output_format': 'markdown', 'mode': 'flash', 'language': 'ch', 'pages': None, 'split_pages': False, 'filename': None}
# 基于历史数据预测未来股价收益

本次竞赛的目标是基于沪深 300指数成分股的历史股价数据，通过建立机器学习模型来预测未来一周收益最大的股票组合（不超过 5只，累计占比不超过1）。选手需通过构建模型、训练和调优，预测并输出未来一周沪深300指数成分股收益最大的股票组合。

## 比赛数据

1) 本次比赛不限制数据格式、内容、范围等，只要所有人都可以免费公开下载使用即可；

2) 大赛基准代码（https://github.com/Sherlock1956/THU-BDC2026）给出的数据为通过 baostock 平台下载的数据；位于data 目录，相关说明详见：get_stock_data_illustration.txt。

## 输出结果

选手的任务是训练模型，基于已有数据，输出预测结果result.csv（UTF-8编码），示例如下：

```csv
stock_id,weight
000408,0.2
000975,0.2
002028,0.2
600372,0.2
600036,0.2
```

1) 带有表头：stock_id,weight；

2) 给出不超过 5个不同的股票代码及权重，每行一个代码及其权重，权重累加和不超过 1，不到1的部分意味着持有现金；

3) 每行都用‘,’分割。

## 三、 评估标准

## 1. 计算收益率：

为了计算这个投资组合从 T+1 开盘买入到 T+5 开盘卖出的总收益率，我们需要先计算出每只股票的单票收益率，然后将它们按权重进行加权求和。由于现金部分通常默认不产生收益（收益率为0），剩余权重即为现金保留。

以下是详细的计算公式：

## 1) 变量定义

n：投资组合中的股票总数 $( \mathtt { n } \le \mathtt { 5 } )$ 。

$\mathbf { W } _ { \mathrm { i } } \mathbf { : }$ 第 i 只股票的分配

# 文本切分

常见的文档切分策略如下：
|策略名称|核心原理|
|--------|--------|
|固定长度切分|按预设字符数或Token数切分|
|递归切分|按优先级分隔符（如段落\n\n > 句子。）逐级递归分割，直至满足大小要求。|
|语义切分|用嵌入模型计算相邻句子相似度，在低于阈值时切分，识别主题转折点。|
|结构感知切分|利用文档元数据（如Markdown/HTML标题）识别逻辑区块进行切分。|
|滑动窗口切分|固定窗口，通过高重叠率（如20%）滑动生成连续分块，保留上下文。|

### TextSplitter
pip install langchain-text-splitters

## 固定长度切分
- 根据字符大小切分
- 根据字节大小切分  

不管哪种都需要用到CharacterTextSplitter这个类。

### CharacterTextSplitter - 按字符切分
- separator : 分隔符，以此作为分隔的基本单元
- chunk_size : 块大小，如果超出则放到下个块
- chunk_overlap : 下一块与上一块重叠的大小，也就是滑动窗口切分

In [11]:
from langchain_text_splitters import CharacterTextSplitter

long_text = docs[0].page_content

# 创建字符切分器
text_splitter = CharacterTextSplitter(
    separator="\n", # 以换行符作为分隔
    chunk_size = 1000, # 每块最大1000字符
    chunk_overlap=200, # 块之间重叠200字符
)

# 切分文本
chunks = text_splitter.split_text(long_text)

print(f"原始文本长度: {len(long_text)} 字符")
print(f"切分为 {len(chunks)} 个块:\n")

for i, chunk in enumerate(chunks):
    print(f"---Chunk{i+1}({len(chunk)}字符)")
    print(chunk)
    print()

原始文本长度: 2482 字符
切分为 3 个块:

---Chunk1(930字符)
# 基于历史数据预测未来股价收益
本次竞赛的目标是基于沪深 300指数成分股的历史股价数据，通过建立机器学习模型来预测未来一周收益最大的股票组合（不超过 5只，累计占比不超过1）。选手需通过构建模型、训练和调优，预测并输出未来一周沪深300指数成分股收益最大的股票组合。
## 比赛数据
1) 本次比赛不限制数据格式、内容、范围等，只要所有人都可以免费公开下载使用即可；
2) 大赛基准代码（https://github.com/Sherlock1956/THU-BDC2026）给出的数据为通过 baostock 平台下载的数据；位于data 目录，相关说明详见：get_stock_data_illustration.txt。
## 输出结果
选手的任务是训练模型，基于已有数据，输出预测结果result.csv（UTF-8编码），示例如下：
```csv
stock_id,weight
000408,0.2
000975,0.2
002028,0.2
600372,0.2
600036,0.2
```
1) 带有表头：stock_id,weight；
2) 给出不超过 5个不同的股票代码及权重，每行一个代码及其权重，权重累加和不超过 1，不到1的部分意味着持有现金；
3) 每行都用‘,’分割。
## 三、 评估标准
## 1. 计算收益率：
为了计算这个投资组合从 T+1 开盘买入到 T+5 开盘卖出的总收益率，我们需要先计算出每只股票的单票收益率，然后将它们按权重进行加权求和。由于现金部分通常默认不产生收益（收益率为0），剩余权重即为现金保留。
以下是详细的计算公式：
## 1) 变量定义
n：投资组合中的股票总数 $( \mathtt { n } \le \mathtt { 5 } )$ 。
$\mathbf { W } _ { \mathrm { i } } \mathbf { : }$ 第 i 只股票的分配权重。
$\mathsf { P } _ { \mathrm { i } , \mathrm { T } + 1 } ^ { \mathrm { o p e n } }$ ：第 i 只股票在 T+1 日的开盘价（买入价）。

---Chunk2(997字符)
$\mathb

### CharacterTextSplitter - 按Token切分
使用OpenAI开源的tiktoken计算token数量，按token数量切分，更精确地控制发送给LLM的token数。

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

# 使用from_tiktoken_encoder，LangChain自带，无需额外安装tiktoken
token_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl1ook_base", # token分词器编码名
    chunk_size = 1000, # 每块最多1000 token
    chunk_overlap = 200 # 块之间重叠200字符
)

chunks = token_splitter.split_text(long_text)

print(f"原始文本长度: {len(long_text)} 字符")
print(f"切分为 {len(chunks)} 个块:\n")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ({len(chunk)}字符) ---")
    print(chunk)
    print()

## 递归字符切分（推荐）
LangChain中提供了一个RecursiveCharacterTextSplitter类，实现了递归字符切分。  
在不超过目标块大小的前提下，尽可能保持段落和句子的完整性。

关键参数如下：
|参数|作用|默认值 (通常情况)|
|----|----|---------------|
|chunk_size|目标块大小 (以字符/token计)。分割器努力让每个块的文本长度不超过这个值。|4000|
|chunk_overlap|块间重叠长度。为了让块与块之间保留一些共同上下文，避免在关键信息处被切断。|200|
|separators|自定义分隔符优先级列表。这是控制分割行为的核心，你可以根据文档格式（如代码、Markdown）调整顺序。|["\n\n", "\n", " ", ""]|
|length_function|长度计算函数。决定用何种方式测量文本长度（例如 len 计字符数，或 tiktoken 计token数）。|len|

其切割流程如下：
1. 输入：原始长文本 T，目标块大小 size，一个有序的字符列表 separators (例如：["\n\n", "\n", "。", " ", ""]，优先级从高到低)。
2. 第一步（用最高级分隔符尝试）：使用当前优先级最高的分隔符（例如段落分隔符 \n\n）尝试将 T 分割成若干块。
3. 检查与判断：
  - 遍历刚分割出的每一个块。
  - 如果这个块的长度小于等于 chunk_size，则将其保留为一个最终的文本块。
  - 如果这个块的长度大于 chunk_size，则不能接受它。
4. 递归降级：对于所有大于 chunk_size 的“超大块”，放弃使用当前分隔符，改用下一个优先级更低的分隔符（例如换行符 \n）来对这个“超大块”再次进行分割。
5. 重复：重复第 3 步和第 4 步，直到所有块都小于等于 chunk_size。
6. 最终手段：如果尝试了所有分隔符，仍然有块大于 chunk_size，那么它会在最后一级分隔符（通常是空字符串 ""，即按字符切分）上，强制将文本按 chunk_size 的长度进行硬截断。

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 创建递归切分器
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 2000,
    chunk_overlap = 200,
    separators=["\n\n", "\n", "。"]  # 优先级从高到低
)

chunks = recursive_splitter.split_documents(long_text)

## 结构感知切分
利用文档元数据识别文档本身的逻辑区块进行切分。

例如：markdown中的多级标题、JSON结构中的字段、Html中的标签等等。  

LangChain都提供了对应不同文档类型的结构感知切分器，例如：
- MarkdownHeaderTextSplitter
- RecursiveJsonSplitter
- HTMLHeaderTextSplitter
- RecursiveCharacterTextSplitter.from_language()

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

# markdown数据
markdown_document = "# 1.Foo\n\n    ## 1.1.Bar\n\nHi this is Jim\n\nHi this is Joe\n\n ### 1.1.1.Boo \n\n Hi this is Lance \n\n ## 1.2.Baz\n\n Hi this is Molly"

# 切分依据，这里是按照三级标题
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

# 创建切分器
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)

chunks = markdown_splitter.split_text(markdown_document)

for doc in chunks:
    # 将 document 转化为 JSON 格式，并且 indent = 2 缩进两格
    print(doc.model_dump_json(indent=2))

{
  "id": null,
  "metadata": {
    "Header 1": "1.Foo",
    "Header 2": "1.1.Bar"
  },
  "page_content": "Hi this is Jim  \nHi this is Joe",
  "type": "Document"
}
{
  "id": null,
  "metadata": {
    "Header 1": "1.Foo",
    "Header 2": "1.1.Bar",
    "Header 3": "1.1.1.Boo"
  },
  "page_content": "Hi this is Lance",
  "type": "Document"
}
{
  "id": null,
  "metadata": {
    "Header 1": "1.Foo",
    "Header 2": "1.2.Baz"
  },
  "page_content": "Hi this is Molly",
  "type": "Document"
}


## 总结
建议切分方式是：
- 优先采用Markdown的文档结构切分，不仅语义完整度高，而且还能记住自己所处的章节
- 当基于文档结构切分的块太大时，可以对超过目标size的块采用递归字符切分，但要保留header到每一个分块

# 向量化  

向量化是将文本转换为高维向量的过程。语义相似的文本在向量空间中距离更近，这是语义检索的基础。

LangChain支持多种Embedding模型平台，可以自由选择：
|模型|提供方|
|----|-----|
|OpenAIEmbeddings|OpenAI|
|DashScopeEmbeddings|阿里云百炼|
|HuggingFaceEmbeddings|本地开源模型|
|OllamaEmbeddings|本地开源模型|

## DashScope Embeddings（阿里云百炼）

In [ ]:
from langchain_community.embeddings import DashScopeEmbeddings
import os

dashscope_embeddings = DashScopeEmbeddings(
    model = "text-embedding-v4",
    dashscope_api_key=os.getenv("DASHSCOPE_API_KEY")
)

# 向量化单条文本
text = "我爱上班"
vector = dashscope_embeddings.embed_query(text)

print(f"文本: {text}")
print(f"向量维度: {len(vector)}")
print(f"向量前5维: {vector[:5]}")


# 批量向量化
texts = ["我要躺平", "我爱工作", "拒绝加班"]
vectors = dashscope_embeddings.embed_documents(texts)
print(f"\n批量向量化: {len(vectors)} 条, 维度: {len(vectors[0])}")

for v in vectors:
    similarity = cosine_similarity(vector, v)
    print("Cosine Similarity:", similarity)

# 向量库

LangChain支持多种向量库：
|向量库|特点|适用场景|
|------|-----|------|
|InMemoryVectorStore|内存存储，零配置|开发测试|
|Chroma|轻量级，支持持久化|中小规模|
|FAISS|高性能，支持GPU|大规模检索|
|Milvus|高性能，支持GPU|大规模检索|

LangChain提供了统一的VectorStore接口，使你可以用统一的方式调用任意向量库：
- add_documents: 添加文档到向量库（不用自己做文本向量化，只要提供好向量模型即可）
- delete: 根据id删除某个文档
- similarity_search: 基于相似度检索与用户问题有关的文档

## 初始化向量库-Chroma

Chroma支持将向量数据持久化到磁盘，适合中小规模应用。
  
安装LangChain的Chroma库：  
pip install langchain-chroma

In [ ]:
from langchain_chroma import Chroma

# 创建向量库
vectorstore = Chroma(
    collection_name="example_collection",  # Chroma 里的"表名"
    embedding_function=dashscope_embeddings,  # embedding函数
    persist_directory="chroma_langchain_db"  # 持久化目录
)

In [ ]:
# 准备文档，我们用之前读取的Markdown文档来测试
with open("./resources/output/r5.md", encoding="utf-8") as f:
    markdown_text = "\n".join(line for line in f.readlines())

# 用递归切分器切分文档
chunks = recursive_splitter.split_documents(
    [Document(page_content=markdown_text, metadata={"filename": "r5.md"})]
)

# 给文档生成id
ids=[]
for i,c in enumerate(chunks):
    c.id = f"doc_{i+1}"
    c.metadata['id'] = c.id
    ids.append(c.id)

# 删除旧文档
vectorstore.delete(ids)

# 添加新文档
vectorstore.add_documents(chunks)

## 检索文档
VectorStore提供了多个检索文档的方法，例如：
- search: 通用搜索方法，支持最多样化的参数
- similarity_search: 基于相似度的搜索
- similarity_search_with_relevance_scores: 基于相似度搜索，并且会返回相似度得分

search方法实现文档检索，其核心参数包括：
- query: 查询条件
- search_type: 查询类型，有3个可选值，
  - similarity:相似度检索，等同于similarity_search
  - similarity_score_threshold:会基于相似度分数阈值做过滤的检索，底层是similarity_search_with_relevance_scores，但不返回得分
  - mmr:先基于相似度检索，再把结果基于mmr算法筛选，提升结果的多样性
其它参数(并不是所有向量库都支持):
- k: 要返回的文档数量（默认值：4）
- score_threshold: similarity_score的最小关联阈值，低于这个分值的文档会被丢弃
- fetch_k: 传递给MMR算法的文档数量（默认：20）
- lambda_mult: MMR返回结果的多样性；1表示最小分集，0表示最大分集。(默认值:0.5)
- filter: 按文档元数据（metadata）筛选

## 相似度检索

In [ ]:
# 用户问题
query = "茅台2025年的市盈率和市净率分别是多少"

# 相似度检索
result = vectorstore.search(
    query=query,
    search_type="similarity",
    k = 5
)

print(f"查询：{query}\n")
for i, doc in enumerate(result):
    print(f"结果{i+1}: {doc.page_content}")
    print(f" 元数据：{doc.metadata}")

### 基于metadata过滤

In [ ]:
# 用户问题
query = "茅台2025年的市盈率和市净率分别是多少"
# 相似度检索
# 先过滤出 id 为 doc_3 的那一条，然后在一条数据里做相似度检索。
results = vectorstore.search(
    query=query,
    search_type="similarity",
    k = 5,
    filter={"id": "doc_3"}
)

print(f"查询: {query}\n")
for i, doc in enumerate(results):
    print(f"结果 {i+1}: {doc.page_content}")
    print(f"  元数据: {doc.metadata}")

### 带相似度得分的检索
调用VectorStore的similarity_search_with_score方法，还可以在检索时返回相似度打分

In [ ]:
# 用户问题
query = "茅台2025年的市盈率和市净率分别是多少"

# 相似度检索
results = vectorstore.similarity_search_with_relevance_scores(
    query=query,
    # search_type="similarity_score_threshold", 不需要search_type
    score_threshold=0.42,  # similarity_score的最小关联阈值，低于这个分值的文档会被丢弃
    k = 5
)

print(f"查询: {query}\n")
for doc, score in results:
    print(f"======文档: {doc.id}，得分：{score}=======")
    print(f"内容: {doc.page_content}")
    print(f"元数据: {doc.metadata}")